In [30]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.metrics import matthews_corrcoef, precision_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split, StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier
import pickle
import matplotlib.pyplot as plt
import re
import itertools
import time
import warnings
warnings.filterwarnings("ignore", module="joblib")
import databento as db
import exchange_calendars as xcals

def close_times():

    # NYSE calendar
    cal = xcals.get_calendar("XNYS")

    # Build schedule for the date range you care about
    #start = "2018-01-01"
    #end = "2030-01-01"
    sched = cal.schedule.loc[:, ["open", "close"]].copy()

    # Convert to America/New_York
    sched["open_et"]  = sched["open"].dt.tz_convert("America/New_York")
    sched["close_et"] = sched["close"].dt.tz_convert("America/New_York")

    # Indicators
    #sched["is_trading_day"] = True
    sched["is_early_close"] = sched["close_et"].dt.time < pd.Timestamp("16:00", tz="America/New_York").time()

    # If you want a per-day close time (minutes since midnight ET)
    sched["session_duration"] = (sched["close_et"].dt.hour * 60 + sched["close_et"].dt.minute) - 9.5 * 60

    # Join to your intraday df by session date
    # assumes df has a Date column that is the NYSE session date (ET)
    sched_out = sched.reset_index().rename(columns={"index": "Date"})
    close_times_df = sched_out
    
    return close_times_df[['Date', 'close_et', 'is_early_close', 'session_duration']]

def add_intraday_labels(df: pd.DataFrame, dt_col: str = "datetime_est") -> pd.DataFrame:
    
    out = df.copy()

    # Ensure datetime
    out[dt_col] = pd.to_datetime(out[dt_col], errors="coerce")
    if out[dt_col].isna().any():
        bad = out[dt_col].isna().sum()
        raise ValueError(f"{bad} rows in {dt_col} could not be parsed to datetime.")

    # Extract time-of-day in minutes since midnight (ET)
    tod_minutes = out[dt_col].dt.hour * 60 + out[dt_col].dt.minute
    out["_tod_minutes"] = tod_minutes.astype(int)

    # Open time (09:30 ET) in minutes
    premarket_min = 7 * 60  # 420
    open_min = 9 * 60 + 30  # 570
    #close_min = 16 * 60 - 1   # 960

    # Minutes since open (can be negative pre-market, positive post-open)
    out["time_to_open"] = out["_tod_minutes"] - open_min
    out["time_to_close"] = (open_min + out["session_duration"]) - out["_tod_minutes"]
    
    out["session_simple"] = "post_market"

    out.loc[out["_tod_minutes"] < premarket_min, "session_simple"] = "overnight"

    out.loc[
        (out["_tod_minutes"] >= premarket_min) & (out["_tod_minutes"] < open_min),
        "session_simple"
    ] = "pre_market"

    out.loc[
        (out["_tod_minutes"] >= open_min) &
        (out["_tod_minutes"] <= open_min + out["session_duration"]),
        "session_simple"
    ] = "open_market"

    # Column 2: detailed session label (your buckets)
    out["session_detail"] = np.select(
        [
            # Pre-market buckets
            (out["_tod_minutes"] < 7 *60),
            (out["_tod_minutes"] >= 7*60) & (out["_tod_minutes"] < 9*60),
            (out["_tod_minutes"] >= 9*60) & (out["_tod_minutes"] < open_min),

            # Open market buckets
            (out["_tod_minutes"] >= open_min) & (out["_tod_minutes"] < 9*60+45),
            (out["_tod_minutes"] >= 9*60+45) & (out["_tod_minutes"] < 10*60),
            (out["_tod_minutes"] >= 10*60) & (out["_tod_minutes"] < 12*60),
            (out["_tod_minutes"] >= 12*60) & (out["_tod_minutes"] < 14*60),
            (out["_tod_minutes"] >= 14*60) & (out["_tod_minutes"] < 15*60+30),
            (out["_tod_minutes"] >= 15*60+30) & (out["_tod_minutes"] < 15*60+45),
            (out["_tod_minutes"] >= 15*60+45) & (out["_tod_minutes"] <= (open_min + out["session_duration"])),

            # Post-market buckets
            (out["_tod_minutes"] > (open_min + out["session_duration"])) & (out["_tod_minutes"] < 16*60+15),
            (out["_tod_minutes"] >= 16*60+15) & (out["_tod_minutes"] < 17*60),
            (out["_tod_minutes"] >= 17*60) & (out["_tod_minutes"] <= 20*60),
        ],
        [
            "overnight",
            "early_pre_market",
            "late_pre_market",
            "early_open",
            "late_open",
            "morning",
            "midday",
            "late_day",
            "early_close",
            "late_close",
            "early_post_market",
            "late_post_market",
            "post_market_other",
        ],
        default="other"
    )

    # Cleanup
    out = out.drop(columns=["_tod_minutes", "_detail_simple_check"], errors="ignore")
    return out

# Read the DBN file into a DBNStore object
dbn_store = db.DBNStore.from_file('qqq_1m.dbn')
# Convert the data to a pandas DataFrame for analysis
df = dbn_store.to_df()
df_main = df.reset_index()[['symbol', 'ts_event', 'close', 'open', 'high', 'low', 'volume']].copy()

# Add in session duration to account for early close on holidays
df_main['datetime_est'] = (df_main['ts_event'].dt.tz_convert('America/New_York'))
df_close_times = close_times()
# Merge close times with intraday data
df_main['Date'] = pd.to_datetime(df_main['datetime_est']).dt.strftime('%Y-%m-%d')
df_close_times["Date"] = pd.to_datetime(df_close_times["Date"]).dt.date
df_main["Date"] = pd.to_datetime(df_main["Date"]).dt.date
df_main = df_main.merge(df_close_times[['Date', 'session_duration']], on="Date", how="left")

# 1. Session Structure & Market Phases

In [31]:
#df_intraday_labels[df_intraday_labels['minutes_since_open'] == 390]
#Shortest minutes_since_open = -330 largest is 629. 0 = 930am, 389 = 4:00pm
df_intraday_labels = add_intraday_labels(df_main)
df_intraday_labels['Date'] = pd.to_datetime(df_intraday_labels['datetime_est']).dt.strftime('%Y-%m-%d')
df_labeled_final = df_intraday_labels[['symbol', 'datetime_est', 'time_to_open', 'time_to_close', 'session_simple', 
                    'session_detail', 'close', 'open', 'high', 'low', 'Date', 'session_duration', 'volume']].copy()

# High Level Feature Engineering

In [281]:
df_features = df_labeled_final.dropna().copy()
df_features = df_features[df_features['session_duration'] == 390] # filter our half days for now
# OC, HL, CH, CL ratios and magnitudes
o, h, l, c = (df_features[k].to_numpy() for k in ("open", "high", "low", "close"))

def dir_th(a, b, pct):
    return (a > (1 + pct) * b).astype(np.int8) - (a < (1 - pct) * b).astype(np.int8)

pairs = {
    "OC": (c, o),
    "HL": (h, l),
    "HC": (h, c),
    "LC": (c, l),
}

for k, (a, b) in pairs.items():
    #df_features[f"{k}_Minute_Direction"] = np.sign(a - b).astype(np.int8)
    df_features[f"{k}_Minute_Magnitude"] = np.round(a / b - 1, 4)
    df_features[f"{k}_Minute_Direction_Low_TH"] = dir_th(a, b, 0.001)
    df_features[f"{k}_Minute_Direction_High_TH"] = dir_th(a, b, 0.01)

In [330]:
# Percent of winning and losing minutes per session_simple and session_detail?
def percent_direction_counts(df, column_to_count, column_to_group):

    df_counts = (
        df
        .assign(
            up   = (df[column_to_count] > 0),
            down = (df[column_to_count] < 0),
            none = (df[column_to_count] == 0),
        )
        .groupby(["Date", column_to_group], sort=False)
        .agg(
            up_minutes=("up", "sum"),
            down_minutes=("down", "sum"),
            none_minutes=("none", "sum"),
        )
    ).reset_index()

    total = df_counts["up_minutes"] + df_counts["down_minutes"]

    df_counts["%_up_minutes"]   = df_counts["up_minutes"]   / total
    df_counts["%_down_minutes"] = df_counts["down_minutes"] / total
    df_counts["%_none_minutes"] = df_counts["none_minutes"] / (total + df_counts["none_minutes"])
    # keep only percent features

    counts = df_counts[
        ["Date", column_to_group, "%_up_minutes", "%_down_minutes", "%_none_minutes"]
    ]

    # pivot wide
    wide = (
        counts
        .pivot(
            index="Date",
            columns=column_to_group,
            values=["%_up_minutes", "%_down_minutes", "%_none_minutes"],
        )
    )

    # flatten MultiIndex columns and prefix with group value
    wide.columns = [
        f"{group}_{metric}"
        for metric, group in wide.columns
    ]

    return wide.fillna(-1).round(3).reset_index()

# Average Volatility between HL 
def magnitude_averages(df, column_to_count='OC_Minute_Magnitude', column_to_group='session_detail'):

    x = df[column_to_count].to_numpy(copy=False)

    df_tmp = df[["Date", column_to_group]].copy()
    df_tmp["pos_val"] = np.where(x > 0, x, np.nan)
    df_tmp["neg_val"] = np.where(x < 0, x, np.nan)

    agg = (
        df_tmp
        .groupby(["Date", column_to_group], sort=False)
        .agg(
            oc_pos_avg=("pos_val", "mean"),
            oc_pos_max=("pos_val", "max"),
            oc_neg_min=("neg_val", "min"),
            oc_neg_avg=("neg_val", "mean"),
        )
    )

    wide = agg.unstack(column_to_group)

    # flatten columns: <group>_<metric>
    wide.columns = [f"{grp}_{metric}" for metric, grp in wide.columns]

    return wide.fillna(0).round(6).reset_index()

column_to_count = 'OC_Minute_Magnitude'
column_to_group = 'session_simple'
session_simple_counts = percent_direction_counts(df_features, column_to_count, column_to_group)

column_to_count = 'OC_Minute_Magnitude'
column_to_group = 'session_detail'
session_detail_counts = percent_direction_counts(df_features, column_to_count, column_to_group)

session_averages = magnitude_averages(df_features)

session_counts_final = pd.merge(session_simple_counts, session_detail_counts, how='inner', on='Date')
session_counts_final = pd.merge(session_counts_final, session_averages, how='inner', on='Date')

,open_market_%_up_minutes,overnight_%_up_minutes_x,post_market_%_up_minutes,pre_market_%_up_minutes,open_market_%_down_minutes,overnight_%_down_minutes_x,post_market_%_down_minutes,pre_market_%_down_minutes,open_market_%_none_minutes,overnight_%_none_minutes_x,...,early_open_oc_neg_avg,late_open_oc_neg_avg,morning_oc_neg_avg,midday_oc_neg_avg,late_day_oc_neg_avg,early_close_oc_neg_avg,late_close_oc_neg_avg,early_post_market_oc_neg_avg,late_post_market_oc_neg_avg,post_market_other_oc_neg_avg
count,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,...,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000,1905.000000
mean,0.505547,0.492275,0.494624,0.491811,0.494453,0.506151,0.505376,0.508189,0.115902,0.535549,...,-0.000692,-0.000571,-0.000447,-0.000338,-0.000355,-0.000372,-0.000461,-0.000284,-0.000183,-0.000191
std,0.029159,0.103343,0.083863,0.062959,0.029159,0.103449,0.083863,0.062959,0.068598,0.187220,...,0.000401,0.000330,0.000236,0.000201,0.000237,0.000295,0.000336,0.000252,0.000142,0.000109
min,0.421000,-1.000000,0.172000,0.167000,0.392000,-1.000000,0.167000,0.214000,0.005000,0.028000,...,-0.004400,-0.002850,-0.003736,-0.002429,-0.002278,-0.005850,-0.004300,-0.002514,-0.002758,-0.001400
25%,0.485000,0.441000,0.439000,0.451000,0.475000,0.464000,0.452000,0.470000,0.072000,0.406000,...,-0.000843,-0.000711,-0.000536,-0.000397,-0.000421,-0.000450,-0.000550,-0.000311,-0.000200,-0.000217
50%,0.506000,0.489000,0.500000,0.491000,0.494000,0.511000,0.500000,0.509000,0.105000,0.545000,...,-0.000600,-0.000500,-0.000395,-0.000286,-0.000287,-0.000289,-0.000371,-0.000200,-0.000144,-0.000165
75%,0.525000,0.536000,0.548000,0.530000,0.515000,0.558000,0.561000,0.549000,0.143000,0.669000,...,-0.000425,-0.000350,-0.000293,-0.000213,-0.000208,-0.000200,-0.000267,-0.000150,-0.000113,-0.000129
max,0.608000,1.000000,0.833000,0.786000,0.579000,1.000000,0.828000,0.833000,0.609000,1.000000,...,0.000000,-0.000100,-0.000122,-0.000100,-0.000100,-0.000100,-0.000100,0.000000,0.000000,0.000000


# Anchored Time Windows

In [332]:
def intraday_aggregations(df, interval, col, name):

    # ensure datetime index
    mask = (df[f"{col}"] >= 0) & (df[f"{col}"] < interval)

    daily_max_min = (
        df.loc[mask]
        .groupby("Date")["close"]
        .agg(lambda x: x.max() / x.min())
        .rename(f"max_min_{name}-{interval}m")
    )
    daily_max_min = pd.DataFrame(daily_max_min).reset_index()

    return daily_max_min

df = df_features.copy()
df_ph = pd.DataFrame()
intervals = [5, 10, 15, 30, 60]
columns = ['time_to_open', 'time_to_close']
names = ['first', 'last']

for interval in intervals:

    for col, name in zip(columns, names):

        daily_max_min = intraday_aggregations(df, interval, col, name)

        if df_ph.empty:
            df_ph = daily_max_min.copy()
        else:
            df_ph = df_ph.merge(daily_max_min, how="left", on="Date")

df_maxmin = df_ph.copy()

In [333]:
df_features_final = pd.merge(session_counts_final, df_maxmin, how='inner', on='Date')
df_features_final

,Date,open_market_%_up_minutes,overnight_%_up_minutes_x,post_market_%_up_minutes,pre_market_%_up_minutes,open_market_%_down_minutes,overnight_%_down_minutes_x,post_market_%_down_minutes,pre_market_%_down_minutes,open_market_%_none_minutes,...,max_min_first-5m,max_min_last-5m,max_min_first-10m,max_min_last-10m,max_min_first-15m,max_min_last-15m,max_min_first-30m,max_min_last-30m,max_min_first-60m,max_min_last-60m
0,2018-05-01,0.540,0.818,0.490,0.500,0.460,0.182,0.510,0.500,0.043,...,1.001994,1.000553,1.003245,1.001045,1.003558,1.003328,1.005493,1.004752,1.007865,1.005932
1,2018-05-02,0.506,0.667,0.391,0.438,0.494,0.333,0.609,0.562,0.074,...,1.001415,1.000742,1.003451,1.001361,1.003451,1.001670,1.004499,1.004020,1.006039,1.008041
2,2018-05-03,0.479,-1.000,0.543,0.469,0.521,-1.000,0.457,0.531,0.043,...,1.001866,1.002291,1.001866,1.002291,1.003363,1.002291,1.006415,1.004705,1.008298,1.004705
3,2018-05-04,0.543,0.200,0.545,0.455,0.457,0.800,0.455,0.545,0.087,...,1.001800,1.001397,1.004532,1.001579,1.005587,1.002035,1.007077,1.002703,1.013223,1.003159
4,2018-05-07,0.533,0.333,0.382,0.434,0.467,0.667,0.618,0.566,0.069,...,1.001993,1.000963,1.003503,1.002291,1.005073,1.002291,1.005797,1.002291,1.005797,1.005074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1900,2025-12-15,0.490,0.590,0.417,0.571,0.510,0.410,0.583,0.429,0.107,...,1.002499,1.000573,1.003444,1.000999,1.003721,1.001458,1.006222,1.001458,1.012788,1.002768
1901,2025-12-16,0.511,0.583,0.487,0.516,0.489,0.417,0.513,0.484,0.105,...,1.001477,1.002617,1.002413,1.003451,1.003533,1.003451,1.006925,1.003451,1.008367,1.004355
1902,2025-12-17,0.455,0.515,0.486,0.515,0.545,0.485,0.514,0.485,0.100,...,1.000735,1.001549,1.003191,1.001549,1.004260,1.003298,1.006271,1.003465,1.006271,1.003615
1903,2025-12-18,0.517,0.627,0.519,0.547,0.483,0.373,0.481,0.453,0.100,...,1.001755,1.001264,1.004012,1.002184,1.005086,1.003596,1.005219,1.003596,1.006075,1.004330


In [334]:
print([c for c in df_features_final.columns if c.endswith("_x")])


['overnight_%_up_minutes_x', 'overnight_%_down_minutes_x', 'overnight_%_none_minutes_x']
